In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
q1 = "Can i still join the course after the start date?"
v1 = model.encode(q1)
v1


array([ 2.13903580e-02, -7.39799663e-02,  1.42067974e-03,  2.13816520e-02,
        2.45113410e-02,  3.15583013e-02, -1.10839710e-01, -1.05017506e-01,
       -6.18259460e-02, -6.42314600e-03,  3.72395804e-03,  9.06393677e-02,
       -9.49936546e-03,  6.53977022e-02,  1.10946996e-02, -2.10097376e-02,
       -3.35125402e-02, -4.31677289e-02,  9.96347889e-03,  1.41969863e-02,
       -6.40415624e-02, -7.04176677e-03, -7.91187733e-02,  5.80030903e-02,
        1.30209664e-03,  4.19731997e-03,  5.70979081e-02,  6.39447942e-02,
        2.49902848e-02, -3.95876281e-02, -3.79506536e-02,  2.70394590e-02,
        1.79423299e-02,  1.72272455e-02,  3.43311243e-02,  9.29057971e-03,
        5.86055145e-02, -4.97789569e-02, -5.05371252e-03,  4.34328578e-02,
       -1.56623218e-02, -2.97534559e-02, -5.13325911e-03,  5.13414517e-02,
        6.16066856e-03,  6.86980486e-02, -1.29505303e-02, -5.61938435e-02,
       -1.08265290e-02,  5.96683510e-02,  5.29940017e-02, -3.42755094e-02,
       -4.15274017e-02, -

In [3]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32332397)

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730553)

In [9]:
v1.dot(v2)

np.float32(-0.14271775)

In [14]:
q3 = "I just discovered the course.Can i still enroll?"
v3 = model.encode(q3)
print(v1.dot(v3))
print(v3.dot(dv))

0.70946705
0.45543757


In [15]:
from ingest import ingest_faq_data
documents = ingest_faq_data()

In [16]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0,len(texts), batch_size)):
    batch = texts[i :i+ batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [18]:
import numpy as np
X = np.array(vectors)

In [19]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [21]:
scores = X.dot(v_query)
scores

array([0.26515424, 0.20766637, 0.08503748, ..., 0.20289946, 0.13746652,
       0.09721454], shape=(1400,), dtype=float32)

In [22]:
idx = np.argmax(scores)
idx,scores[idx]

(np.int64(510), np.float32(0.76294106))

In [23]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [24]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([ 510, 1153,   77,  905,  515])

In [25]:
scores[top5]

array([0.76294106, 0.7579371 , 0.7192133 , 0.6536312 , 0.56009996],
      dtype=float32)

In [27]:
top5 = np.argsort(-scores)[:5]
top5

array([ 510, 1153,   77,  905,  515])

In [28]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192133
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [29]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X,documents)

In [30]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results = 5)

In [31]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [32]:
results = vindex.search(
    query_vector,
    filter_dict= {"course":"llm-zoomcamp"},
    num_results=5
)

In [33]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [35]:
from rag_helper import RAGBase
from google import genai
from google.genai import types
from dotenv import load_dotenv
from ingest import build_index

SYSTEM_INSTRUCTION = """
Your task is to answer the questions from the course participants based on the given context
    Use the context to find relevant information and provide accurate answer to the user
    if there is no answer in the context do not make up things just tell the user that you don't know
"""

load_dotenv()
client = genai.Client()
index = build_index(documents)
config = types.GenerateContentConfig(
    system_instruction= SYSTEM_INSTRUCTION
)

assistant = RAGBase(
    index = index,
    llm_client = client,
    config = config,
    model = "gemini-3.6-flash"
)

question = "I just found out about the program, can i still sign up?"

In [48]:
assistant.rag(question)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 4.18429641s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, '

In [42]:
class RAGvector(RAGBase):
    def __init__(self,embedder,**kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self,query,num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results= num_results,
            filter_dict = filter_dict
        )

In [46]:
vector_assistant = RAGvector(
    embedder=model,
    index=vindex,
    llm_client=client,
    config = config,
    model = "gemini-3.6-flash"
)

In [47]:
vector_assistant.rag(question)

'Yes, you can still join. If you want to receive a certificate, you just need to submit your project while submissions are still being accepted. \n\nYou can simply start learning and submitting homework (while the form is open) without officially registering, as registration is just used to gauge interest before the start date and is not checked against a registered list.'